In [4]:
import ROOT
import numpy as np
import ctypes

# 1. Setup
f = ROOT.TFile.Open("tumor.root")
tree = f.Get("t")
energy_vec = ROOT.std.vector('double')()
vlm_vec = ROOT.std.vector('int')()
tree.SetBranchAddress("et", energy_vec)
tree.SetBranchAddress("vlm", vlm_vec)

# Detectors are at 10 units away
det_pos = {4: (10,0,0), 6: (-10,0,0), 5: (0,-10,0)}
h3 = ROOT.TH3F("h3", "Tumor Mapping;X;Y;Z", 100, -10, 10, 100, -10, 10, 100, -10, 10)

# 2. Scattering Reconstruction Loop
for i in range(tree.GetEntries()):
    tree.GetEntry(i)
    for j in range(energy_vec.size()):
        e_scattered = energy_vec[j]
        v_id = vlm_vec[j]
        
        # Energy check: scattered gammas will have lower energy than the beam
        if 0.10 < e_scattered < 0.70: 
            if v_id in det_pos:
                p_det = det_pos[v_id]
                
                # We project a ray from the detector into the phantom 
                # to find where it crosses the incoming beam path
                for t in np.linspace(0, 20, 200):
                    # For detectors 4 (+X) and 6 (-X), the path moves toward the Y-axis
                    # We trace back from detector toward the center
                    step = t / 10.0
                    x_trace = p_det[0] * (1 - step)
                    y_trace = p_det[1] + (step * (0 - p_det[1])) # Moves toward beam
                    z_trace = p_det[2] * (1 - step)
                    
                    h3.Fill(x_trace, y_trace, z_trace)

# 3. Finding the Hotspot (The Tumor)
max_bin = h3.GetMaximumBin()
tx, ty, tz = ctypes.c_int(), ctypes.c_int(), ctypes.c_int()
h3.GetBinXYZ(max_bin, tx, ty, tz)

print(f"Tumor Scatter Point detected at Y = {h3.GetYaxis().GetBinCenter(ty.value):.2f}")

# 4. Visualization
c = ROOT.TCanvas("c", "Tumor Map", 800, 600)
h3.SetMinimum(h3.GetMaximum() * 0.5) # Show only the high-density scatter zone
h3.SetFillColor(ROOT.kRed)
h3.Draw("BOX2")
c.SaveAs("tumor_map.png")

Tumor Scatter Point detected at Y = 0.10


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c
Info in <TCanvas::Print>: png file tumor_map.png has been created
